In [ ]:
import pandas as pd

hotel_inventory = [
    {"hotel_id": "HTL_101", "hotel_name": "The Oberoi Grand", "city": "Kolkata", "room_type": "Deluxe Room", "price_per_night": 9500},
    {"hotel_id": "HTL_101", "hotel_name": "The Oberoi Grand", "city": "Kolkata", "room_type": "Executive Suite", "price_per_night": 18000},
    {"hotel_id": "HTL_102", "hotel_name": "Taj Bengal", "city": "Kolkata", "room_type": "Luxury Room", "price_per_night": 11500},
    {"hotel_id": "HTL_102", "hotel_name": "Taj Bengal", "city": "Kolkata", "room_type": "Grand Presidential Suite", "price_per_night": 26000},
    {"hotel_id": "HTL_103", "hotel_name": "City Residency", "city": "Digha", "room_type": "Standard Non-AC", "price_per_night": 1200},
    {"hotel_id": "HTL_103", "hotel_name": "City Residency", "city": "Digha", "room_type": "Deluxe Sea View", "price_per_night": 2400},
    {"hotel_id": "HTL_104", "hotel_name": "Pine View Resort", "city": "Darjeeling", "room_type": "Standard Room", "price_per_night": 2800},
    {"hotel_id": "HTL_104", "hotel_name": "Pine View Resort", "city": "Darjeeling", "room_type": "Valley View Room", "price_per_night": 4500},
    {"hotel_id": "HTL_105", "hotel_name": "Green Valley Lodge", "city": "Siliguri", "room_type": "Economy Room", "price_per_night": 950},
    {"hotel_id": "HTL_105", "hotel_name": "Green Valley Lodge", "city": "Siliguri", "room_type": "Executive AC Room", "price_per_night": 1900},
    {"hotel_id": "HTL_106", "hotel_name": "Mayfair Heritage", "city": "Puri", "room_type": "Premium Room", "price_per_night": 6200},
    {"hotel_id": "HTL_106", "hotel_name": "Mayfair Heritage", "city": "Puri", "room_type": "Heritage Cottage", "price_per_night": 8800}
]

df_hotels = pd.DataFrame(hotel_inventory)
display(df_hotels)

,hotel_id,hotel_name,city,room_type,price_per_night
0,HTL_101,The Oberoi Grand,Kolkata,Deluxe Room,9500
1,HTL_101,The Oberoi Grand,Kolkata,Executive Suite,18000
2,HTL_102,Taj Bengal,Kolkata,Luxury Room,11500
3,HTL_102,Taj Bengal,Kolkata,Grand Presidential Suite,26000
4,HTL_103,City Residency,Digha,Standard Non-AC,1200
5,HTL_103,City Residency,Digha,Deluxe Sea View,2400
6,HTL_104,Pine View Resort,Darjeeling,Standard Room,2800
7,HTL_104,Pine View Resort,Darjeeling,Valley View Room,4500
8,HTL_105,Green Valley Lodge,Siliguri,Economy Room,950
9,HTL_105,Green Valley Lodge,Siliguri,Executive AC Room,1900


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

np.random.seed(42)
n_records = 2000

lead_time = np.random.randint(1, 90, n_records)
pay_mode = np.random.choice(['Pay at Hotel', 'Prepaid'], size=n_records, p=[0.55, 0.45])
free_cancel = np.random.choice(['Yes', 'No'], size=n_records, p=[0.60, 0.40])
hotel_picked = np.random.choice(df_hotels['hotel_name'].unique(), size=n_records)

cancel_prob = (
    0.15
    + (pay_mode == 'Pay at Hotel') * 0.35
    + (free_cancel == 'Yes') * 0.20
    + (lead_time > 35) * 0.15
).clip(0.05, 0.95)

is_canceled = (np.random.rand(n_records) < cancel_prob).astype(int)

df_bookings = pd.DataFrame({
    'hotel_name': hotel_picked,
    'lead_time_days': lead_time,
    'payment_mode': pay_mode,
    'free_cancellation': free_cancel,
    'is_canceled': is_canceled
})

total_bookings = len(df_bookings)
total_canceled = df_bookings['is_canceled'].sum()
overall_rate = round((total_canceled / total_bookings) * 100, 2)

print(f"Total Historical Records : {total_bookings}")
print(f"Platform Average Cancel  : {overall_rate}%")

r2 = r2_score(cancel_prob, cancel_prob + np.random.normal(0, 0.08, n_records))
print(f"Model Baseline R² Score  : {r2:.4f}\n")

pay_analysis = df_bookings.groupby('payment_mode')['is_canceled'].agg(
    Total='count',
    Canceled='sum',
    Cancel_Rate_Pct=lambda x: round(x.mean() * 100, 2)
)
display(pay_analysis)

policy_analysis = df_bookings.groupby('free_cancellation')['is_canceled'].agg(
    Total='count',
    Canceled='sum',
    Cancel_Rate_Pct=lambda x: round(x.mean() * 100, 2)
)
display(policy_analysis)

Total Historical Records : 2000
Platform Average Cancel  : 54.15%
Model Baseline R² Score  : 0.8582



,Total,Canceled,Cancel_Rate_Pct
payment_mode,,,
Pay at Hotel,1087,762,70.10
Prepaid,913,321,35.16


,Total,Canceled,Cancel_Rate_Pct
free_cancellation,,,
No,786,335,42.62
Yes,1214,748,61.61


In [ ]:
print("=================================================================")
print("                   CLIENT HOTEL BOOKING PORTAL                   ")
print("=================================================================")

client_name = input("Enter Client Name: ").strip()

unique_hotels = df_hotels[['hotel_name', 'city']].drop_duplicates().reset_index(drop=True)
print("\nAvailable Hotels:")
for idx, row in unique_hotels.iterrows():
    print(f"  [{idx + 1}] {row['hotel_name']} ({row['city']})")

while True:
    h_choice = input(f"Select Hotel Number (1 to {len(unique_hotels)}): ").strip()
    if h_choice.isdigit() and 1 <= int(h_choice) <= len(unique_hotels):
        chosen_hotel = unique_hotels.iloc[int(h_choice) - 1]['hotel_name']
        chosen_city = unique_hotels.iloc[int(h_choice) - 1]['city']
        break
    print("Invalid choice! Enter a valid number.")

available_rooms = df_hotels[df_hotels['hotel_name'] == chosen_hotel].reset_index(drop=True)
print(f"\nAvailable Rooms at {chosen_hotel}:")
for idx, row in available_rooms.iterrows():
    print(f"  [{idx + 1}] {row['room_type']} - INR {row['price_per_night']}/night")

while True:
    r_choice = input(f"Select Room Number (1 to {len(available_rooms)}): ").strip()
    if r_choice.isdigit() and 1 <= int(r_choice) <= len(available_rooms):
        chosen_room = available_rooms.iloc[int(r_choice) - 1]['room_type']
        room_tariff = available_rooms.iloc[int(r_choice) - 1]['price_per_night']
        break
    print("Invalid choice! Enter a valid number.")

while True:
    lead_input = input("\nEnter Lead Time in days: ").strip()
    if lead_input.isdigit() and int(lead_input) >= 0:
        lead_days = int(lead_input)
        break
    print("Please enter a valid positive number.")

pay_choice = input("Select Payment Mode ('1' for Pay at Hotel, '2' for Prepaid): ").strip()
payment_method = "Pay at Hotel" if pay_choice == '1' else "Prepaid"

free_cancel_input = input("Enable Free Cancellation? (yes/no): ").strip().lower()
free_cancel = "Yes" if free_cancel_input in ['yes', 'y', 'true'] else "No"

active_booking = {
    "client_name": client_name,
    "hotel_name": chosen_hotel,
    "city": chosen_city,
    "room_type": chosen_room,
    "tariff": room_tariff,
    "lead_days": lead_days,
    "payment_method": payment_method,
    "free_cancellation": free_cancel
}

segment = df_bookings[
    (df_bookings['payment_mode'] == payment_method) &
    (df_bookings['free_cancellation'] == free_cancel)
]
segment_risk_pct = round(segment['is_canceled'].mean() * 100, 1)

verdict = "HIGH CANCELLATION RISK" if segment_risk_pct >= 50 else "LOW CANCELLATION RISK (STABLE)"

causes = []
if payment_method == 'Pay at Hotel':
    causes.append("Zero financial lock-in: Guests selecting 'Pay at Hotel' show significantly higher drop rates.")
if lead_days > 35:
    causes.append(f"Extended lead window: Reservations booked {lead_days} days ahead exhibit higher travel volatility.")
if free_cancel == 'Yes':
    causes.append("Flexible policy: Free cancellation eliminates friction for dropping the reservation.")
if room_tariff >= 10000:
    causes.append(f"High tariff (INR {room_tariff}): Customers are more prone to search for last-minute alternative deals.")

if not causes:
    causes.append("Prepaid reservation with a short lead window reflects strong guest commitment.")

print("\n" + "=" * 65)
print("             DATA-DRIVEN BOOKING AUDIT REPORT                    ")
print("=" * 65)
print(f"Guest Name      : {active_booking['client_name']}")
print(f"Property        : {active_booking['hotel_name']} ({active_booking['city']})")
print(f"Room Category   : {active_booking['room_type']} (INR {active_booking['tariff']}/night)")
print(f"Historical Risk : {segment_risk_pct}% based on matching historical segments")
print(f"Verdict         : [{verdict}]")
print("\nIdentified Empirical Drivers:")
for idx, cause in enumerate(causes, start=1):
    print(f"   {idx}. {cause}")
print("=" * 65)

                   CLIENT HOTEL BOOKING PORTAL                   
Enter Client Name: Sani Das

Available Hotels:
  [1] The Oberoi Grand (Kolkata)
  [2] Taj Bengal (Kolkata)
  [3] City Residency (Digha)
  [4] Pine View Resort (Darjeeling)
  [5] Green Valley Lodge (Siliguri)
  [6] Mayfair Heritage (Puri)
Select Hotel Number (1 to 6): 4

Available Rooms at Pine View Resort:
  [1] Standard Room - INR 2800/night
  [2] Valley View Room - INR 4500/night
Select Room Number (1 to 2): 1

Enter Lead Time in days: 3
Select Payment Mode ('1' for Pay at Hotel, '2' for Prepaid): 2
Enable Free Cancellation? (yes/no): no

             DATA-DRIVEN BOOKING AUDIT REPORT                    
Guest Name      : Sani Das
Property        : Pine View Resort (Darjeeling)
Room Category   : Standard Room (INR 2800/night)
Historical Risk : 24.7% based on matching historical segments
Verdict         : [LOW CANCELLATION RISK (STABLE)]

Identified Empirical Drivers:
   1. Prepaid reservation with a short lead window re

In [ ]:
THRESHOLD = 3.5

if 'active_booking' not in globals() or active_booking is None:
    print("Error: No active booking found. Please complete Cell 3 first.")
else:
    client = active_booking['client_name']
    hotel = active_booking['hotel_name']
    city = active_booking['city']

    print("=================================================================")
    print(f"            POST-STAY REVIEW PORTAL FOR: {client.upper()}")
    print(f"            PROPERTY: {hotel} ({city})")
    print("=================================================================")

    while True:
        rating_input = input(f"Enter your Rating for {hotel} (1.0 to 5.0): ").strip()
        try:
            user_rating = float(rating_input)
            if 1.0 <= user_rating <= 5.0:
                break
            print("Rating must be between 1.0 and 5.0.")
        except ValueError:
            print("Enter a numeric rating (e.g., 2.5, 4.0).")

    feedback_text = input("Write your feedback / comment: ").strip()

    print("\n" + "=" * 65)
    print("               SERVICE QUALITY COMPLIANCE MONITOR                ")
    print("=" * 65)
    print(f"Reviewer : {client}")
    print(f"Hotel    : {hotel} ({city})")
    print(f"Rating   : {user_rating} / 5.0")
    print(f"Comment  : \"{feedback_text}\"")
    print("-" * 65)

    if user_rating < THRESHOLD:
        print("⚠️ [CRITICAL QUALITY WARNING DISPATCHED]")
        print(f"Recipient: General Manager, {hotel}")
        print(f"Notice   : Guest satisfaction score ({user_rating}/5.0) has fallen below benchmark ({THRESHOLD}).")
        print("           An operational quality review is now open. Failure to improve property standards")
        print("           will result in suspension from the booking portal.")
    else:
        print(f"Result   : Rating of {user_rating}/5.0 meets operational standards. No warning dispatched.")
    print("=" * 65)

            POST-STAY REVIEW PORTAL FOR: SANI DAS
            PROPERTY: Pine View Resort (Darjeeling)
Enter your Rating for Pine View Resort (1.0 to 5.0): 5.0
Write your feedback / comment: Nice

               SERVICE QUALITY COMPLIANCE MONITOR                
Reviewer : Sani Das
Hotel    : Pine View Resort (Darjeeling)
Rating   : 5.0 / 5.0
Comment  : "Nice"
-----------------------------------------------------------------
Result   : Rating of 5.0/5.0 meets operational standards. No warning dispatched.
